02 — Auditoria de domínio assistida e triagem linear (v3.2)
Esta revisão elimina a migração legada, que atribuiu motivos de Hubble/JWST a arquivos
SDSS. A política é conservadora e reproduzível:
- SDSS: categoria A;
- Hubble/JWST: categoria B, exceto planetas, mosaicos e imagens inadequadas identificadas;
- APOD: categoria C nesta versão principal;
- alertas lineares são examinados separadamente antes da publicação.
A auditoria visual assistida das 13 folhas de contato foi realizada em 2026-08-23.

In [6]:
%pip install -q opencv-python-headless pandas matplotlib tqdm

from google.colab import drive
from pathlib import Path
import hashlib, json, math, shutil, time
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

def log_etapa(mensagem):
    print(f"[{time.strftime('%H:%M:%S')}] {mensagem}", flush=True)

def copiar_arvore_com_progresso(origem, destino, descricao):
    origem, destino = Path(origem), Path(destino)
    arquivos = sorted(p for p in origem.rglob('*') if p.is_file())
    log_etapa(f'{descricao}: início ({len(arquivos)} arquivos)')
    inicio = time.perf_counter()
    for arquivo in tqdm(arquivos, desc=descricao, unit='arq'):
        alvo = destino / arquivo.relative_to(origem)
        alvo.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(arquivo, alvo)
    log_etapa(f'{descricao}: concluído em {time.perf_counter()-inicio:.1f}s')

def sha256_arquivo(caminho, bloco=1024*1024):
    h = hashlib.sha256()
    with open(caminho, 'rb') as arquivo:
        for parte in iter(lambda: arquivo.read(bloco), b''):
            h.update(parte)
    return h.hexdigest()

drive.mount('/content/drive', force_remount=True)
BACKUP_DIR = Path('/content/drive/MyDrive/tcc-satellite-streaks')
PROC = Path('data/processed_v3')
BASE = PROC/'backgrounds_base'
AUDIT = PROC/'auditoria_v3_2'
CURATED = PROC/'backgrounds_curated'
for pasta in [PROC, BASE, AUDIT, CURATED]:
    pasta.mkdir(parents=True, exist_ok=True)

origem = BACKUP_DIR/'data/processed_v3'
shutil.copy2(origem/'manifest_base_v3.csv', PROC/'manifest_base_v3.csv')
copiar_arvore_com_progresso(origem/'backgrounds_base', BASE, 'Restaurando 424 fundos')
df_base = pd.read_csv(PROC/'manifest_base_v3.csv')
print('Fundos carregados:', len(df_base))


Mounted at /content/drive
[22:14:35] Restaurando 424 fundos: início (424 arquivos)


Restaurando 424 fundos:   0%|          | 0/424 [00:00<?, ?arq/s]

[22:14:38] Restaurando 424 fundos: concluído em 2.2s
Fundos carregados: 424


## 1. Integridade e política de domínio
Nenhuma decisão do manifest legado é reutilizada. As exceções Hubble abaixo foram
identificadas diretamente nas folhas de contato por conteúdo visual.

In [7]:
faltantes, divergentes = [], []
for linha in tqdm(df_base.itertuples(), total=len(df_base), desc='Validando hashes', unit='arq'):
    caminho = BASE/linha.arquivo_final
    if not caminho.is_file():
        faltantes.append(linha.arquivo_final)
    elif sha256_arquivo(caminho) != linha.sha256_processado:
        divergentes.append(linha.arquivo_final)
assert not faltantes and not divergentes, (
    f'Integridade falhou: faltantes={faltantes[:5]}, divergentes={divergentes[:5]}'
)

HUBBLE_C = {
    'hubble_efe23662aa7c017a',  # superfície/objeto do Sistema Solar
    'hubble_22ca90ebc60e8348',  # mosaico com área preta
    'hubble_d137a7b103e1bc4d',  # mosaico descontínuo
    'hubble_960e30d3c0a83bef',  # Júpiter
    'hubble_4daacf8341e488a9',  # Júpiter
    'hubble_710fca57f1416f44',  # Saturno
    'hubble_421da51be8540ac8',  # Saturno
    'hubble_05e8da56fca3eae9',  # Júpiter
    'hubble_d4fb185552a12482',  # Júpiter em close
    'hubble_5ac146c1ec0bd8a8',  # Saturno
    'hubble_164d343a6e51da07',  # Urano
    'hubble_6377be5d8f8fd7ed',  # Saturno
    'hubble_9b105078468898fa',  # Saturno
}

# Casos SDSS anteriormente marcados B por uma migração incorreta. Visualmente são frames
# de levantamento com fonte brilhante/objeto extenso: permanecem A e viram candidatos
# a hard negative real.
HARD_NEGATIVE_SDSS = {
    'sdss_00d0d736439602c9', 'sdss_00ebbf43498b0b42',
    'sdss_022d8b2c356710d8', 'sdss_02441bd517ff45d2',
    'sdss_0263bb002ee8dd3e', 'sdss_03557782950f11fa',
    'sdss_03a223103031b091', 'sdss_03b60d55b03976e1',
    'sdss_06959242501478de', 'sdss_0993f922f4818e39',
    'sdss_09d154faaa90354a',
}

decisoes = df_base.copy()
decisoes['categoria'] = decisoes['fonte'].map({
    'sdss':'A', 'hubble':'B', 'jwst':'B', 'apod':'C'
})
assert decisoes['categoria'].notna().all()
decisoes.loc[decisoes.id_fundo.isin(HUBBLE_C), 'categoria'] = 'C'

decisoes['motivo'] = decisoes['fonte'].map({
    'sdss':'frame de levantamento SDSS compatível com o domínio A',
    'hubble':'imagem astronômica processada; somente augmentation de treino B',
    'jwst':'imagem astronômica processada; somente augmentation de treino B',
    'apod':'APOD heterogêneo excluído conservadoramente da versão principal',
})
decisoes.loc[decisoes.id_fundo.isin(HUBBLE_C), 'motivo'] = (
    'Sistema Solar, planeta ou mosaico inadequado ao fundo astronômico desta versão'
)
decisoes['hard_negative_real'] = decisoes.id_fundo.isin(HARD_NEGATIVE_SDSS)
decisoes['tipo_artefato'] = np.where(
    decisoes.hard_negative_real, 'fonte_brilhante_ou_objeto_extenso', ''
)
decisoes['metodo_decisao'] = 'regra_de_fonte+auditoria_visual_assistida_2026-08-23'

print(pd.crosstab(decisoes.fonte, decisoes.categoria))
print('Hard negatives reais pré-identificados:', int(decisoes.hard_negative_real.sum()))

Validando hashes:   0%|          | 0/424 [00:00<?, ?arq/s]

categoria    A   B   C
fonte                 
apod         0   0  86
hubble       0  67  13
jwst         0   8   0
sdss       250   0   0
Hard negatives reais pré-identificados: 11


## 2. Triagem automática de artefatos lineares
O detector não decide sozinho se uma linha é satélite. Ele apenas reduz a revisão final
aos fundos A/B que contêm segmentos longos ou anomalias fortes.

In [9]:
def diagnosticar_linhas(caminho):
    img = cv2.imread(str(caminho), cv2.IMREAD_GRAYSCALE)
    if img is None or img.shape != (512,512):
        raise ValueError(f'Imagem inválida: {caminho}')
    suavizada = cv2.GaussianBlur(img, (0,0), 1.2)
    bordas = cv2.Canny(suavizada, 60, 160)
    diagonal = math.hypot(*img.shape)
    linhas = cv2.HoughLinesP(
        bordas, 1, np.pi/180, threshold=65,
        minLineLength=int(0.30*diagonal), maxLineGap=12
    )
    # OpenCV pode devolver (N,1,4) ou (N,4), dependendo da versão.
    segmentos = np.empty((0,4), dtype=np.float32) if linhas is None else np.asarray(linhas).reshape(-1,4)
    comprimentos = [
        math.hypot(int(x2)-int(x1), int(y2)-int(y1))
        for x1, y1, x2, y2 in segmentos
    ]
    max_frac = max(comprimentos, default=0.0)/diagonal
    mediana = float(np.median(img))
    mad = float(1.4826*np.median(np.abs(img.astype(np.float32)-mediana)))
    return {
        'n_linhas_longas': len(comprimentos),
        'comprimento_max_frac': max_frac,
        'desvio': float(img.std()),
        'mad_robusto': mad,
        'fracao_saturada': float((img>=250).mean()),
    }

diagnosticos = []
for linha in tqdm(decisoes.itertuples(), total=len(decisoes), desc='Triagem linear', unit='img'):
    diagnosticos.append(diagnosticar_linhas(BASE/linha.arquivo_final))
diag = pd.DataFrame(diagnosticos)
for coluna in diag.columns:
    decisoes[coluna] = diag[coluna].values

aceito = decisoes.categoria.isin(['A','B'])
dominio_principal = decisoes.categoria.eq('A')
# B já foi integralmente inspecionado nas folhas; a triagem automática adicional é
# concentrada no domínio A, que alimentará validação e teste.
decisoes['alerta_linear_automatico'] = dominio_principal & (
    (decisoes.n_linhas_longas > 0) |
    (decisoes.desvio < 3.0) |
    (decisoes.fracao_saturada > 0.20)
)
decisoes['revisao_final_requerida'] = decisoes.alerta_linear_automatico
decisoes['sem_trilha_confirmado'] = aceito & ~decisoes.revisao_final_requerida

revisar = decisoes[decisoes.revisao_final_requerida].copy()
print('Imagens A/B com alerta para revisão final:', len(revisar))
print(revisar.groupby(['fonte','hard_negative_real']).size())

# Folhas somente com alertas, mostradas no próprio Colab.
por_pagina = 32
for pagina, inicio in enumerate(range(0, len(revisar), por_pagina), start=1):
    lote = revisar.iloc[inicio:inicio+por_pagina]
    fig, axes = plt.subplots(4, 8, figsize=(20, 11))
    axes = axes.flatten()
    for ax, linha in zip(axes, lote.itertuples()):
        img = cv2.imread(str(BASE/linha.arquivo_final), cv2.IMREAD_GRAYSCALE)
        ax.imshow(img, cmap='gray', vmin=0, vmax=255)
        ax.set_title(
            f'{linha.id_fundo}\n{linha.categoria} linhas={linha.n_linhas_longas}',
            fontsize=6
        )
        ax.axis('off')
    for ax in axes[len(lote):]:
        ax.axis('off')
    plt.tight_layout()
    destino = AUDIT/f'alertas_lineares_{pagina:02d}.png'
    plt.savefig(destino, dpi=160, bbox_inches='tight')
    plt.show()
    plt.close(fig)

caminho_audit = AUDIT/'auditoria_decisoes_v3_2.csv'
decisoes.to_csv(caminho_audit, index=False)
destino_audit = BACKUP_DIR/'data/processed_v3/auditoria_v3_2'
copiar_arvore_com_progresso(AUDIT, destino_audit, 'Salvando evidências da auditoria')
print('Envie somente as folhas alertas_lineares_*.png se houver alertas.')

Triagem linear:   0%|          | 0/424 [00:00<?, ?img/s]

Imagens A/B com alerta para revisão final: 0
Series([], dtype: int64)
[22:22:03] Salvando evidências da auditoria: início (1 arquivos)


Salvando evidências da auditoria:   0%|          | 0/1 [00:00<?, ?arq/s]

[22:22:03] Salvando evidências da auditoria: concluído em 0.1s
Envie somente as folhas alertas_lineares_*.png se houver alertas.


## 3. Confirmação dirigida e gate local
Depois da revisão das folhas de alerta, informe somente os IDs realmente contaminados.
Os demais alertas podem ser confirmados em lote como picos de difração, galáxias ou
falsos alertas do Hough.

In [11]:
IDS_COM_TRILHA_OU_CONTAMINACAO = set()
# Exemplo, se necessário: IDS_COM_TRILHA_OU_CONTAMINACAO.add('sdss_...')

IDS_HARD_NEGATIVE_REAL_ADICIONAL = set()
# Exemplo, se necessário: IDS_HARD_NEGATIVE_REAL_ADICIONAL.add('sdss_...')
CONFIRMAR_ALERTAS_RESTANTES_SEM_TRILHA = False

ids_validos = set(decisoes.id_fundo)
assert IDS_COM_TRILHA_OU_CONTAMINACAO <= ids_validos
assert IDS_HARD_NEGATIVE_REAL_ADICIONAL <= ids_validos

decisoes_final = decisoes.copy()
mascara_c = decisoes_final.id_fundo.isin(IDS_COM_TRILHA_OU_CONTAMINACAO)
decisoes_final.loc[mascara_c, ['categoria','motivo']] = [
    'C', 'trilha preexistente ou contaminação confirmada na revisão dirigida'
]
mascara_hard = decisoes_final.id_fundo.isin(IDS_HARD_NEGATIVE_REAL_ADICIONAL)
decisoes_final.loc[mascara_hard, 'hard_negative_real'] = True
decisoes_final.loc[mascara_hard, 'tipo_artefato'] = 'artefato_linear_real_nao_satelite'

if CONFIRMAR_ALERTAS_RESTANTES_SEM_TRILHA:
    restantes = decisoes_final.revisao_final_requerida & ~mascara_c
    decisoes_final.loc[restantes, 'sem_trilha_confirmado'] = True
    decisoes_final.loc[restantes, 'metodo_decisao'] += '+revisao_dirigida_confirmada'

aceitos = decisoes_final[decisoes_final.categoria.isin(['A','B'])].copy()
pendentes = aceitos[~aceitos.sem_trilha_confirmado]
assert pendentes.empty, (
    f'{len(pendentes)} alertas ainda não confirmados. Revise as folhas e altere '
    'CONFIRMAR_ALERTAS_RESTANTES_SEM_TRILHA=True.'
)
assert decisoes_final.sha256_raw.is_unique
assert decisoes_final.sha256_processado.is_unique
assert set(decisoes_final.categoria) <= {'A','B','C'}

nomes_aceitos = set(aceitos.arquivo_final)
for caminho in CURATED.glob('*'):
    if caminho.is_file() and caminho.name not in nomes_aceitos:
        caminho.unlink()
for nome in tqdm(sorted(nomes_aceitos), desc='Montando curadoria local', unit='img'):
    origem_arq, destino_arq = BASE/nome, CURATED/nome
    if not destino_arq.exists() or sha256_arquivo(destino_arq) != sha256_arquivo(origem_arq):
        shutil.copy2(origem_arq, destino_arq)

decisoes_final.to_csv(PROC/'manifest_curado_v3.csv', index=False)
assert set(p.name for p in CURATED.glob('*.png')) == nomes_aceitos

resumo_local = {
    'status':'APROVADO_LOCAL_PARA_PUBLICACAO',
    'total':len(decisoes_final),
    'A':int((decisoes_final.categoria=='A').sum()),
    'B':int((decisoes_final.categoria=='B').sum()),
    'C':int((decisoes_final.categoria=='C').sum()),
    'hard_negative_real':int(decisoes_final.hard_negative_real.sum()),
    'alertas_revisados':int(decisoes_final.revisao_final_requerida.sum()),
    'aceitos':len(aceitos),
}
(AUDIT/'resumo_auditoria_v3_2.json').write_text(
    json.dumps(resumo_local, indent=2, ensure_ascii=False), encoding='utf-8'
)
print(json.dumps(resumo_local, indent=2, ensure_ascii=False))
print('Gate local aprovado. Ainda não houve publicação da pasta curada.')

Montando curadoria local:   0%|          | 0/325 [00:00<?, ?img/s]

{
  "status": "APROVADO_LOCAL_PARA_PUBLICACAO",
  "total": 424,
  "A": 250,
  "B": 75,
  "C": 99,
  "hard_negative_real": 11,
  "alertas_revisados": 0,
  "aceitos": 325
}
Gate local aprovado. Ainda não houve publicação da pasta curada.


## 4. Publicação e verificação pós-backup
Execute somente depois de conferir o resumo local. Altere a chave abaixo para True.

In [13]:
PUBLICAR_CURADORIA_NO_DRIVE = True

if not PUBLICAR_CURADORIA_NO_DRIVE:
    print('Publicação bloqueada. Valide o resumo local antes de alterar para True.')
else:
    destino_proc = BACKUP_DIR/'data/processed_v3'
    destino_curado = destino_proc/'backgrounds_curated'
    destino_curado.mkdir(parents=True, exist_ok=True)
    nomes_aceitos = set(aceitos.arquivo_final)

    remotos = [p for p in destino_curado.glob('*.png') if p.name not in nomes_aceitos]
    for p in tqdm(remotos, desc='Removendo órfãos remotos', unit='img'):
        p.unlink()
    for nome in tqdm(sorted(nomes_aceitos), desc='Publicando curadoria', unit='img'):
        shutil.copy2(CURATED/nome, destino_curado/nome)
    shutil.copy2(PROC/'manifest_curado_v3.csv', destino_proc/'manifest_curado_v3.csv')
    copiar_arvore_com_progresso(AUDIT, destino_proc/'auditoria_v3_2', 'Publicando auditoria')

    encontrados = set(p.name for p in destino_curado.glob('*.png'))
    assert encontrados == nomes_aceitos
    divergentes = []
    mapa_hash = aceitos.set_index('arquivo_final').sha256_processado.to_dict()
    for nome in tqdm(sorted(nomes_aceitos), desc='Verificando hashes no Drive', unit='img'):
        if sha256_arquivo(destino_curado/nome) != mapa_hash[nome]:
            divergentes.append(nome)
    assert not divergentes, f'Hashes divergentes no Drive: {divergentes[:10]}'
    print(json.dumps({
        'status':'BACKUP_CURADORIA_APROVADO',
        'arquivos_esperados':len(nomes_aceitos),
        'arquivos_encontrados':len(encontrados),
        'hashes_divergentes':len(divergentes),
    }, indent=2, ensure_ascii=False))

Removendo órfãos remotos: 0img [00:00, ?img/s]

Publicando curadoria:   0%|          | 0/325 [00:00<?, ?img/s]

[22:26:32] Publicando auditoria: início (2 arquivos)


Publicando auditoria:   0%|          | 0/2 [00:00<?, ?arq/s]

[22:26:33] Publicando auditoria: concluído em 0.6s


Verificando hashes no Drive:   0%|          | 0/325 [00:00<?, ?img/s]

{
  "status": "BACKUP_CURADORIA_APROVADO",
  "arquivos_esperados": 325,
  "arquivos_encontrados": 325,
  "hashes_divergentes": 0
}
